# 03 — Cargas, clusters y waveforms promedio

Estudios de carga (Qtotal, Qfast/Qslow), clusters de forma de pulso y waveform
promedio por canal con ajuste por deconvolución sobre la selección de calidad.
La lógica vive en `pmtcheck_charge.py` y `pmtcheck_average.py`.

**Canales a analizar:** 14, 12, 37, 0, 17, 10, 16, 2, 7, 40, 6, 36, 34, 46, 24,
32, 42, 26, 22, 44, 30, 20 (orden descendente y de izq a dch).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
from waffles.data_classes.WaveformSet import WaveformSet

from pmtcheck_io import load_wfset
from pmtcheck_quality import build_quality_wfset
import pmtcheck_charge as charge
import pmtcheck_average as average
import pmtcheck_viewers as viewers

In [ ]:
run = 43363
wfset = load_wfset(run)

# Mismos cortes por defecto que el notebook 02 (pmtcheck_config).
wfset_quality, qres = build_quality_wfset(wfset)

## Qtotal versus amplitude para un canal

In [ ]:
cc16 = charge.channel_charges(wfset_quality, endpoint=110, channel=16)
charge.plot_qtotal_vs_amplitude(cc16)

## Qfast versus Qslow para un canal

In [ ]:
cc14 = charge.channel_charges(wfset_quality, endpoint=110, channel=14)
charge.plot_qfast_vs_qslow(cc14)

In [ ]:
charge.plot_ratio_vs_qslow(cc14)

## Clusters de forma de pulso (Qfast/Qslow)

In [ ]:
# Rangos definidos especificamente para el canal 14
qslow_range = (75_000, 175_000)
qfast_range = (0, 12_500)

clusters = charge.split_clusters(cc14, qslow_range=qslow_range, qfast_range=qfast_range)
charge.plot_cluster_averages(cc14, clusters)

In [ ]:
charge.plot_cluster_scatter(cc14, qslow_range=qslow_range)

## Visor de waveforms individuales de cada cluster

`plot_cluster_event` funciona siempre; `browse_cluster_events` usa
ipywidgets y puede quedarse ejecutando en VS Code — si pasa, interrúmpela
y usa la versión sin widgets.

In [ ]:
viewers.plot_cluster_event(clusters, "deviating", 0)   # cluster_name: "deviating" / "rest"

In [ ]:
# Version con widgets (opcional, ver nota de arriba)
# viewers.browse_cluster_events(clusters)

## Waveform promedio + fit de un solo canal

In [ ]:
def select_ch16(wf):
    return wf.endpoint == 110 and wf.channel == 16

wfset_ch16 = WaveformSet.from_filtered_WaveformSet(wfset_quality, select_ch16, show_progress=False)

plt.figure(figsize=(8, 5))
deconv = average.pmt_average_and_fit(wfset_ch16)
plt.show()

## PMT layout: waveform promedio y fit en la posición física de cada PMT

In [ ]:
average_wvf = {}
fig, axs = average.plot_pmt_layout(
    wfset_quality,
    endpoint=110,
    average_store=average_wvf,
    savepath="pmts_average_waveforms.png",
)